# 🌟 图像分类端到端项目实战

**项目目标**：使用 PyTorch 构建一个完整的图像分类模型，从数据准备到模型部署

**数据集**：CIFAR-10（小样本图像分类经典数据集）

**预期成果**：构建一个准确率 >85% 的 CNN 分类器

---

## 📋 目录
1. [环境准备与依赖安装](#1-环境准备)
2. [数据准备与增强](#2-数据准备)
3. [模型设计与实现](#3-模型设计)
4. [训练与调参](#4-训练与调参)
5. [模型评估与错误分析](#5-模型评估)
6. [模型保存与加载](#6-模型保存)
7. [可视化分析](#7-可视化)

## 1. 环境准备与依赖安装 <a name="1-环境准备"></a>

In [ ]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import random
import os
import time
from datetime import datetime

# 设置随机种子确保可复现性
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 检查GPU是否可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 使用设备: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 数据准备与增强 <a name="2-数据准备"></a>

In [ ]:
# CIFAR-10 类别定义
CLASSES = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

NUM_CLASSES = 10
print(f"📦 数据集: CIFAR-10 ({NUM_CLASSES} 类)")
print(f"   类别: {CLASSES}")

In [ ]:
# 数据增强策略
# 训练集：随机裁剪、水平翻转、颜色抖动、归一化
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),           # 随机裁剪
    transforms.RandomHorizontalFlip(p=0.5),        # 随机水平翻转
    transforms.ColorJitter(brightness=0.2,          # 颜色抖动
                          contrast=0.2,
                          saturation=0.2,
                          hue=0.1),
    transforms.RandomRotation(15),                   # 随机旋转
    transforms.ToTensor(),                           # 转为 Tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465),  # 归一化
                         (0.2470, 0.2435, 0.2616))
])

# 测试集：仅归一化
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

print("✅ 数据增强策略已定义")
print("   训练集增强: 随机裁剪 + 水平翻转 + 颜色抖动 + 随机旋转")
print("   测试集增强: 仅归一化")

In [ ]:
# 下载并加载数据集
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

# 划分验证集 (90% 训练, 10% 验证)
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# 创建数据加载器
BATCH_SIZE = 128
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"\n📊 数据集划分:")
print(f"   训练集: {len(train_dataset)} 样本")
print(f"   验证集: {len(val_dataset)} 样本")
print(f"   测试集: {len(test_dataset)} 样本")
print(f"\n📦 Batch Size: {BATCH_SIZE}")
print(f"   训练批次数: {len(train_loader)}")
print(f"   验证批次数: {len(val_loader)}")

In [ ]:
# 可视化部分样本和数据增强效果
def imshow(img, title=None):
    """反归一化并显示图像"""
    img = img / 2 + 0.5     # 反归一化
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title)
    plt.axis('off')

# 显示原始样本
sample_images, sample_labels = next(iter(test_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()
for i in range(10):
    img, label = sample_images[i], sample_labels[i].item()
    axes[i].imshow(np.transpose(img.numpy() / 2 + 0.5, (1, 2, 0)))
    axes[i].set_title(CLASSES[label])
    axes[i].axis('off')
plt.suptitle('CIFAR-10 样本展示', fontsize=14)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 模型设计与实现 <a name="3-模型设计"></a>

In [ ]:
# 方法1: 自定义 CNN 模型

class BasicCNN(nn.Module):
    """
    基础卷积神经网络
    结构: Conv -> BN -> ReLU -> MaxPool (重复3次) -> FC -> FC
    """
    def __init__(self, num_classes=10):
        super(BasicCNN, self).__init__()
        
        # 第一个卷积块
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)  # 32 -> 16
        )
        
        # 第二个卷积块
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)  # 16 -> 8
        )
        
        # 第三个卷积块
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)  # 8 -> 4
        )
        
        # 全连接层
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

# 创建模型
model = BasicCNN(num_classes=NUM_CLASSES).to(device)
print("🏗️  BasicCNN 模型结构:")
print(model)

# 计算模型参数量
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
print(f"\n📊 模型参数量: {total_params:,} ({total_params/1e6:.2f}M)")

In [ ]:
# 方法2: 使用预训练模型 (ResNet18) - 迁移学习

class TransferResNet(nn.Module):
    """
    基于预训练 ResNet18 的迁移学习模型
    修改最后一层全连接层以适应 CIFAR-10
    """
    def __init__(self, num_classes=10, pretrained=True):
        super(TransferResNet, self).__init__()
        
        # 加载预训练 ResNet18
        self.resnet = models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        
        # 修改第一层以接受 32x32 输入 (CIFAR-10)
        # 原设计是为 224x224 图像
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()  # 移除原始的 MaxPool
        
        # 修改最后一层全连接层
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.resnet(x)

# 创建迁移学习模型
model_resnet = TransferResNet(num_classes=NUM_CLASSES, pretrained=True).to(device)
print("🏗️  TransferResNet 模型结构 (部分):")
print(f"   输入层: Conv2d(3, 64, kernel_size=3, stride=1)")
print(f"   移除 MaxPool: True")
print(f"   输出层: Linear(512, {NUM_CLASSES})")
print(f"\n📊 参数量: {count_parameters(model_resnet):,} ({count_parameters(model_resnet)/1e6:.2f}M)")

## 4. 训练与调参 <a name="4-训练与调参"></a>

In [ ]:
# 训练配置
class TrainConfig:
    def __init__(self):
        self.epochs = 50
        self.learning_rate = 0.001
        self.weight_decay = 1e-4  # L2正则化
        self.momentum = 0.9
        self.patience = 10  # 早停耐心值
        
        # 学习率调度
        self.scheduler_type = 'cosine'  # 'step', 'cosine', 'plateau'
        self.step_size = 20  # StepLR 步长
        self.step_gamma = 0.1  # StepLR 衰减率
        
        # 优化器选择
        self.optimizer_type = 'adam'  # 'sgd', 'adam', 'adamw'

config = TrainConfig()
print("⚙️  训练配置:")
print(f"   轮次: {config.epochs}")
print(f"   学习率: {config.learning_rate}")
print(f"   权重衰减: {config.weight_decay}")
print(f"   早停耐心值: {config.patience}")
print(f"   学习率调度: {config.scheduler_type}")

In [ ]:
# 早停机制
class EarlyStopping:
    """
    早停机制: 当验证损失连续多次不下降时停止训练
    """
    def __init__(self, patience=10, min_delta=0.001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif self._is_improvement(score):
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"\n🛑 早停触发! 验证集性能已连续 {self.patience} 轮未改善")
                
    def _is_improvement(self, score):
        if self.mode == 'min':
            return score < self.best_score - self.min_delta
        else:
            return score > self.best_score + self.min_delta

print("✅ 早停机制已定义")

In [ ]:
# 训练函数
def train_epoch(model, train_loader, criterion, optimizer, device):
    """训练一个epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # 前向传播
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # 进度显示
        if (batch_idx + 1) % 50 == 0:
            print(f"      Batch [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, val_loader, criterion, device):
    """验证模型"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    val_loss = running_loss / len(val_loader)
    val_acc = 100. * correct / total
    return val_loss, val_acc

print("✅ 训练和验证函数已定义")

In [ ]:
# 主训练循环
def train_model(model, train_loader, val_loader, config, device, model_name='model'):
    """完整训练流程"""
    
    # 损失函数
    criterion = nn.CrossEntropyLoss()
    
    # 优化器
    if config.optimizer_type == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=config.learning_rate,
                             momentum=config.momentum,
                             weight_decay=config.weight_decay)
    elif config.optimizer_type == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=config.learning_rate,
                              weight_decay=config.weight_decay)
    else:  # adamw
        optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate,
                               weight_decay=config.weight_decay)
    
    # 学习率调度器
    if config.scheduler_type == 'step':
        scheduler = optim.lr_scheduler.StepLR(optimizer, 
                                               step_size=config.step_size,
                                               gamma=config.step_gamma)
    elif config.scheduler_type == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, 
                                                        T_max=config.epochs)
    else:  # plateau
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 
                                                         mode='min', 
                                                         factor=0.5, 
                                                         patience=5)
    
    # 早停
    early_stopping = EarlyStopping(patience=config.patience)
    
    # 训练历史
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'learning_rate': []
    }
    
    best_val_acc = 0.0
    best_model_path = f'{model_name}_best.pth'
    
    print(f"\n🚀 开始训练 {model_name}")
    print("=" * 70)
    
    for epoch in range(config.epochs):
        epoch_start = time.time()
        
        # 训练
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # 验证
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # 学习率调度
        if config.scheduler_type == 'plateau':
            scheduler.step(val_loss)
        else:
            scheduler.step()
        
        current_lr = optimizer.param_groups[0]['lr']
        
        # 记录历史
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['learning_rate'].append(current_lr)
        
        epoch_time = time.time() - epoch_start
        
        # 打印结果
        print(f"Epoch [{epoch+1:2d}/{config.epochs}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
              f"LR: {current_lr:.6f} | Time: {epoch_time:.1f}s")
        
        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'config': config.__dict__
            }, best_model_path)
            print(f"      ✅ 保存最佳模型 (Val Acc: {val_acc:.2f}%)")
        
        # 早停检查
        early_stopping(-val_acc)  # 注意取负，因为早停是最小化
        if early_stopping.early_stop:
            print(f"\n🛑 早停于 Epoch {epoch+1}")
            break
    
    print("=" * 70)
    print(f"🏆 训练完成! 最佳验证准确率: {best_val_acc:.2f}%")
    
    return history, best_model_path

print("✅ 主训练循环已定义")

In [ ]:
# 训练 BasicCNN 模型
print("=" * 70)
print("🔵 实验1: 训练 BasicCNN (从头训练)")
print("=" * 70)

# 使用较小配置快速演示
config_demo = TrainConfig()
config_demo.epochs = 20  # 演示用减少轮次
config_demo.learning_rate = 0.003

model_basic = BasicCNN(num_classes=NUM_CLASSES).to(device)
history_basic, best_path_basic = train_model(
    model_basic, train_loader, val_loader, 
    config_demo, device, 
    model_name='basic_cnn'
)

In [ ]:
# 训练 TransferResNet 模型
print("\n" + "=" * 70)
print("🟢 实验2: 训练 TransferResNet (迁移学习)")
print("=" * 70)

# 迁移学习使用较小学习率
config_transfer = TrainConfig()
config_transfer.epochs = 20
config_transfer.learning_rate = 0.0005  # 预训练模型用较小学习率

model_transfer = TransferResNet(num_classes=NUM_CLASSES, pretrained=True).to(device)
history_transfer, best_path_transfer = train_model(
    model_transfer, train_loader, val_loader,
    config_transfer, device,
    model_name='resnet18_transfer'
)

## 5. 模型评估与错误分析 <a name="5-模型评估"></a>

In [ ]:
# 加载最佳模型进行测试
def test_model(model, test_loader, device):
    """在测试集上评估模型"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    accuracy = 100. * correct / total
    return np.array(all_preds), np.array(all_labels), np.array(all_probs), accuracy

# 测试 BasicCNN
checkpoint_basic = torch.load(best_path_basic, map_location=device)
model_basic.load_state_dict(checkpoint_basic['model_state_dict'])
preds_basic, labels_basic, probs_basic, acc_basic = test_model(
    model_basic, test_loader, device
)
print(f"🔵 BasicCNN 测试准确率: {acc_basic:.2f}%")

# 测试 TransferResNet
checkpoint_transfer = torch.load(best_path_transfer, map_location=device)
model_transfer.load_state_dict(checkpoint_transfer['model_state_dict'])
preds_transfer, labels_transfer, probs_transfer, acc_transfer = test_model(
    model_transfer, test_loader, device
)
print(f"🟢 TransferResNet 测试准确率: {acc_transfer:.2f}%")

In [ ]:
# 混淆矩阵分析
def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix'):
    """绘制混淆矩阵"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{title.lower().replace(" ", "_")}.png', 
                dpi=150, bbox_inches='tight')
    plt.show()
    
    return cm

print("📊 TransferResNet 混淆矩阵:")
cm_transfer = plot_confusion_matrix(
    labels_transfer, preds_transfer, CLASSES,
    title='TransferResNet'
)

In [ ]:
# 分类报告
print("📋 分类详细报告 (TransferResNet):")
print(classification_report(labels_transfer, preds_transfer, 
                           target_names=CLASSES,
                           digits=4))

In [ ]:
# 错误分析：找出最容易被误分类的样本
def analyze_misclassifications(model, test_loader, device, classes, n_samples=10):
    """分析误分类样本"""
    model.eval()
    
    misclassified = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            probs, predicted = outputs.max(1)
            
            # 找出误分类的样本
            mask = ~predicted.eq(targets)
            
            for i in range(len(mask)):
                if mask[i]:
                    misclassified.append({
                        'image': inputs[i].cpu(),
                        'true_label': classes[targets[i].item()],
                        'pred_label': classes[predicted[i].item()],
                        'confidence': probs[i].item()
                    })
    
    return misclassified

# 获取误分类样本
errors = analyze_misclassifications(model_transfer, test_loader, device, CLASSES)
print(f"\n🔍 共发现 {len(errors)} 个误分类样本")

# 可视化部分误分类样本
def plot_errors(errors, classes, n=10):
    """可视化误分类样本"""
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for i, error in enumerate(errors[:n]):
        img = error['image']
        img = img / 2 + 0.5  # 反归一化
        
        axes[i].imshow(np.transpose(img.numpy(), (1, 2, 0)))
        axes[i].set_title(f"True: {error['true_label']}\nPred: {error['pred_label']}\nConf: {error['confidence']:.2f}")
        axes[i].axis('off')
    
    plt.suptitle('误分类样本分析', fontsize=14)
    plt.tight_layout()
    plt.savefig('misclassified_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_errors(errors, CLASSES)

In [ ]:
# 每类的准确率分析
def per_class_accuracy(y_true, y_pred, classes):
    """计算每个类别的准确率"""
    cm = confusion_matrix(y_true, y_pred)
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    
    plt.figure(figsize=(12, 5))
    bars = plt.bar(classes, per_class_acc * 100, color='steelblue')
    plt.xlabel('类别')
    plt.ylabel('准确率 (%)')
    plt.title('各类别准确率 (TransferResNet)')
    plt.xticks(rotation=45)
    plt.ylim(0, 105)
    
    # 添加数值标签
    for bar, acc in zip(bars, per_class_acc):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return dict(zip(classes, per_class_acc))

class_acc = per_class_accuracy(labels_transfer, preds_transfer, CLASSES)
print("\n📊 各类别准确率:")
for cls, acc in sorted(class_acc.items(), key=lambda x: x[1]):
    print(f"   {cls:>8s}: {acc*100:5.2f}%")

## 6. 模型保存与加载 <a name="6-模型保存"></a>

In [ ]:
# 模型保存的多种方式

# 方式1: 保存整个模型 (不推荐，包含模型结构)
# torch.save(model, 'model_full.pth')

# 方式2: 保存模型参数 (推荐，跨平台兼容性好)
torch.save(model_transfer.state_dict(), 'model_weights.pth')
print("✅ 方式2: 模型权重已保存至 'model_weights.pth'")

# 方式3: 保存完整检查点 (推荐，包含训练状态)
torch.save({
    'epoch': 20,
    'model_state_dict': model_transfer.state_dict(),
    'optimizer_state_dict': optim.Adam(model_transfer.parameters()).state_dict(),
    'val_acc': acc_transfer,
    'config': config_transfer.__dict__
}, 'model_checkpoint.pth')
print("✅ 方式3: 完整检查点已保存至 'model_checkpoint.pth'")

In [ ]:
# 模型加载

# 加载模型权重
model_loaded = TransferResNet(num_classes=NUM_CLASSES, pretrained=False).to(device)
model_loaded.load_state_dict(torch.load('model_weights.pth', map_location=device))
model_loaded.eval()

# 验证加载后的模型准确率
_, _, _, acc_loaded = test_model(model_loaded, test_loader, device)
print(f"✅ 模型加载验证 - 测试准确率: {acc_loaded:.2f}%")

# 加载检查点
checkpoint = torch.load('model_checkpoint.pth', map_location=device)
print(f"\n📋 检查点信息:")
print(f"   训练轮次: {checkpoint['epoch']}")
print(f"   验证准确率: {checkpoint['val_acc']:.2f}%")
print(f"   学习率: {checkpoint['config']['learning_rate']}")

## 7. 可视化分析 <a name="7-可视化"></a>

In [ ]:
# 训练过程可视化
def plot_training_history(histories, model_names):
    """绘制训练历史"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = ['#1f77b4', '#2ca02c']
    
    for i, (history, name) in enumerate(zip(histories, model_names)):
        color = colors[i]
        
        # 损失曲线
        axes[0, 0].plot(history['train_loss'], color=color, 
                       linestyle='-', label=f'{name} Train', alpha=0.8)
        axes[0, 0].plot(history['val_loss'], color=color, 
                       linestyle='--', label=f'{name} Val', alpha=0.8)
        
        # 准确率曲线
        axes[0, 1].plot(history['train_acc'], color=color, 
                       linestyle='-', label=f'{name} Train', alpha=0.8)
        axes[0, 1].plot(history['val_acc'], color=color, 
                       linestyle='--', label=f'{name} Val', alpha=0.8)
        
        # 学习率曲线
        axes[1, 0].plot(history['learning_rate'], color=color, 
                       linestyle='-', label=name, alpha=0.8)
    
    axes[0, 0].set_title('Training & Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].set_title('Training & Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].set_title('Learning Rate Schedule')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_yscale('log')
    
    # 准确率对比
    final_accs = [h['val_acc'][-1] for h in histories]
    axes[1, 1].bar(model_names, final_accs, color=colors)
    axes[1, 1].set_title('Final Validation Accuracy')
    axes[1, 1].set_ylabel('Accuracy (%)')
    axes[1, 1].set_ylim(0, 100)
    for i, acc in enumerate(final_accs):
        axes[1, 1].text(i, acc + 1, f'{acc:.1f}%', ha='center')
    
    plt.tight_layout()
    plt.savefig('training_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history([history_basic, history_transfer], 
                     ['BasicCNN', 'TransferResNet'])

In [ ]:
# 模型性能对比总结
print("\n" + "=" * 70)
print("📊 模型性能对比总结")
print("=" * 70)
print(f"\n{'模型':<20} {'测试准确率':<15} {'参数量':<15}")
print("-" * 50)
print(f"{'BasicCNN':<20} {acc_basic:.2f}%{'':<8} {count_parameters(model_basic)/1e6:.2f}M")
print(f"{'TransferResNet':<20} {acc_transfer:.2f}%{'':<8} {count_parameters(model_transfer)/1e6:.2f}M")
print("-" * 50)

improvement = acc_transfer - acc_basic
print(f"\n🎯 TransferResNet 相比 BasicCNN 提升: {improvement:+.2f}%")
print(f"   迁移学习利用 ImageNet 预训练知识，显著提升效果")

---

## 📝 项目总结

### 主要成果
1. **构建了完整的图像分类项目流程**：从数据加载、模型设计到训练评估
2. **实现了两种模型对比**：
   - BasicCNN：从头训练，适合理解 CNN 基础结构
   - TransferResNet：迁移学习，效果显著更好
3. **集成了多种训练技巧**：
   - 数据增强（随机裁剪、翻转、颜色抖动）
   - BatchNorm 批归一化
   - Dropout 正则化
   - 学习率调度（CosineAnnealing）
   - 早停机制
4. **完善的可视化分析**：
   - 混淆矩阵
   - 训练曲线
   - 误分类分析
   - 各类别准确率

### 下一步优化方向
1. **模型架构**：尝试 ResNet50、EfficientNet 等更深网络
2. **数据增强**：尝试 CutMix、MixUp、AutoAugment 等高级增强
3. **训练策略**：尝试 Label Smoothing、学习率预热
4. **模型集成**：多模型融合提升泛化能力

### 相关资源
- 项目优化指南: `项目优化指南.md`
- 模型部署入门: `模型部署入门.md`

---